In [1]:
import os
os.environ["HADOOP_HOME"] = r"D:\hadoop"
os.environ["hadoop.home.dir"] = r"D:\hadoop"

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("HousePriceETL") \
    .master("local[*]") \
    .config("spark.driver.memory", "6g") \
    .config("spark.executor.memory", "6g") \
    .config("spark.sql.shuffle.partitions", "64") \
    .config(
        "spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.3.4,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.262,"
        "com.microsoft.sqlserver:mssql-jdbc:10.2.3.jre8"
    ) \
    .config("spark.hadoop.fs.s3a.endpoint", "http://127.0.0.1:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "12345678") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

print("Spark Version:", spark.version)

Spark Version: 3.5.1


Check data mogi_data

In [3]:
df = spark.read.csv(
    "s3a://housing-data/bronze/mogi_data.csv",
    header=True,
    inferSchema=True
)

df.show(5)
df.printSchema()

+--------------------+--------------------+--------------+-----------------+--------------+---------+-------+--------------+----------+--------+--------------------+
|               Title|      Địa chỉ cụ thể|           Giá|Diện tích sử dụng| Diện tích đất|Phòng ngủ|Nhà tắm|       Pháp lý| Ngày đăng|  Mã BĐS|                Link|
+--------------------+--------------------+--------------+-----------------+--------------+---------+-------+--------------+----------+--------+--------------------+
|Nhà HXH Nguyễn Vă...|Nguyễn Văn Khối, ...|7 tỷ 800 triệu|            90 m2|  90 m2 (5x18)|      7.0|    8.0|Không xác định|23/03/2022|21514724|https://mogi.vn/q...|
|Xe hơi đỗ trong n...|Lê Văn Khương, Ph...|5 tỷ 300 triệu|            80 m2|  80 m2 (4x20)|      2.0|    3.0|Không xác định|23/03/2022|21515660|https://mogi.vn/q...|
|nhà hẻm xe tải, 5...|Phan Huy Ích, Phư...|7 tỷ 500 triệu|            60 m2|  60 m2 (4x15)|      5.0|    4.0|       Sổ hồng|23/03/2022|21515962|https://mogi.vn/q...|
|Nhà

Đặt lại tên cột 

In [4]:
from pyspark.sql.functions import col
df = (
    df
    .withColumnRenamed("Title", "title")
    .withColumnRenamed("Địa chỉ cụ thể", "address")
    .withColumnRenamed("Giá", "price")
    .withColumnRenamed("Diện tích sử dụng", "usable_area")
    .withColumnRenamed("Diện tích đất", "land_area")
    .withColumnRenamed("Phòng ngủ", "bedrooms")
    .withColumnRenamed("Nhà tắm", "bathrooms")
    .withColumnRenamed("Pháp lý", "legal_status")
    .withColumnRenamed("Ngày đăng", "posted_date")
    .withColumnRenamed("Mã BĐS", "property_id")
    .withColumnRenamed("Link", "url")
)

df.printSchema()

root
 |-- title: string (nullable = true)
 |-- address: string (nullable = true)
 |-- price: string (nullable = true)
 |-- usable_area: string (nullable = true)
 |-- land_area: string (nullable = true)
 |-- bedrooms: string (nullable = true)
 |-- bathrooms: string (nullable = true)
 |-- legal_status: string (nullable = true)
 |-- posted_date: string (nullable = true)
 |-- property_id: string (nullable = true)
 |-- url: string (nullable = true)



Xử lý col Price

In [5]:
df.select("price").distinct().orderBy("price").show(100, truncate=False)

+-------------------------------------------------+
|price                                            |
+-------------------------------------------------+
|NULL                                             |
|                                                 |
|  Q9                                             |
| 19.3 Tỷ"                                        |
| 1T1L nhà mới 8 tỷ / 97 m2"                      |
| 2.53tỷ Full"                                    |
| 3 lầu chí  21.5 tỷ."                            |
| 3PN                                             |
| 4 tầng                                         |
| 4 tầng                                          |
| 5 tầng                                          |
| 5 tầng BTCT                                     |
| 5m x 18m đường Lê Trọng Tấn giá chào 7/2 tỷ TL."|
| 68m2                                            |
| 69m2                                            |
| 7x20m                                           |
| 80m2      

Xóa Null 

In [6]:
from pyspark.sql.functions import col
df = df.filter(col("price").isNotNull())

In [7]:
from pyspark.sql.functions import sum, when
df.select(
    sum(when(col("price").isNull(), 1).otherwise(0)).alias("null_price")
).show()

+----------+
|null_price|
+----------+
|         0|
+----------+



Xóa các cột không liên quan

In [8]:
from pyspark.sql.functions import col, lower

df = df.filter(
    col("price").isNotNull() &
    (~lower(col("price")).contains("thỏa thuận")) &
    (~lower(col("price")).contains("liên hệ")) &
    (col("price") != "999.000 đ")
)

Chuẩn hóa về VND

In [9]:
from pyspark.sql.functions import (
    col,
    regexp_replace,
    regexp_extract,
    when,
    lit
)

# Chuẩn hóa dấu phẩy thành dấu chấm
df = df.withColumn(
    "price",
    regexp_replace(col("price"), ",", ".")
)

# Trích xuất phần tỷ và triệu
df = (
    df
    .withColumn(
        "billion",
        regexp_extract(col("price"), r'(\d+(?:\.\d+)?)\s*tỷ', 1)
    )
    .withColumn(
        "million",
        regexp_extract(col("price"), r'(\d+(?:\.\d+)?)\s*triệu', 1)
    )
)

# Chuyển sang kiểu số
df = (
    df
    .withColumn(
        "billion",
        when(col("billion") == "", lit(0))
        .otherwise(col("billion").cast("double"))
    )
    .withColumn(
        "million",
        when(col("million") == "", lit(0))
        .otherwise(col("million").cast("double"))
    )
)

# Tính giá theo VNĐ
df = df.withColumn(
    "price",
    (
        col("billion") * 1_000_000_000 +
        col("million") * 1_000_000
    ).cast("long")
)

# Xóa cột tạm
df = df.drop("billion", "million")

In [10]:
df = df.filter(col("price") > 0)

In [11]:
df.select("price").show(10, False)

+----------+
|price     |
+----------+
|7800000000|
|5300000000|
|7500000000|
|4700000000|
|5700000000|
|5800000000|
|6800000000|
|4800000000|
|6900000000|
|5600000000|
+----------+
only showing top 10 rows



In [12]:
df.count()

500755

Chỉ đang chuẩn hóa về dạng nm.000.000.000 ( đơn giá tiền chưa xử lý outlier)

Xử lý địa chỉ address

In [13]:
from pyspark.sql.functions import (
    split, size, element_at, when, trim, col
)

# Tách địa chỉ theo dấu phẩy
parts = split(col("address"), ",")

# Các phần từ cuối
part1 = trim(element_at(parts, -1))   # City
part2 = trim(element_at(parts, -2))   # District
part3 = trim(element_at(parts, -3))   # Ward hoặc Street
part4 = trim(element_at(parts, -4))   # Street (nếu có Ward)

# Regex nhận diện Ward/Xã/Thị trấn
ward_regex = r"(?i)^(Phường|P\.?\s*|Xã|X\.?\s*|Thị\s+Trấn)\b"

df = (
    df
    # City
    .withColumn(
        "city",
        when(size(parts) >= 1, part1)
    )

    # District
    .withColumn(
        "district",
        when(size(parts) >= 2, part2)
    )

    # Ward
    .withColumn(
        "ward",
        when(
            (size(parts) >= 3) & part3.rlike(ward_regex),
            part3
        ).otherwise(None)
    )

    # Street
    .withColumn(
        "street",
        when(
            size(parts) >= 4,
            when(
                part3.rlike(ward_regex),
                part4          # Có ward -> street là phần thứ 4 từ cuối
            ).otherwise(part3) # Không có ward -> part3 chính là street
        ).when(
            (size(parts) == 3) & (~part3.rlike(ward_regex)),
            part3              # Địa chỉ chỉ có Đường, Quận, TP
        ).otherwise(None)
    )
)

In [14]:
df.select(
    "address",
    "street",
    "ward",
    "district",
    "city"
).show(20, truncate=False)

+----------------------------------------------------------+---------------+----------------------+---------------+-----+
|address                                                   |street         |ward                  |district       |city |
+----------------------------------------------------------+---------------+----------------------+---------------+-----+
|Nguyễn Văn Khối, Phường 11, Quận Gò Vấp, TPHCM            |Nguyễn Văn Khối|Phường 11             |Quận Gò Vấp    |TPHCM|
|Lê Văn Khương, Phường Thới An, Quận 12, TPHCM             |Lê Văn Khương  |Phường Thới An        |Quận 12        |TPHCM|
|Phan Huy Ích, Phường 12, Quận Gò Vấp, TPHCM               |Phan Huy Ích   |Phường 12             |Quận Gò Vấp    |TPHCM|
|Nguyễn Văn Khối, Phường 8, Quận Gò Vấp, TPHCM             |Nguyễn Văn Khối|Phường 8              |Quận Gò Vấp    |TPHCM|
|Tô Ngọc Vân, Phường Thạnh Xuân, Quận 12, TPHCM            |Tô Ngọc Vân    |Phường Thạnh Xuân     |Quận 12        |TPHCM|
|Phạm Văn Chiêu, Phường 

Chuẩn hóa city thành TP.HCM

In [15]:
from pyspark.sql.functions import col, when, trim
df = df.withColumn(
    "city",
    trim(col("city"))
)
df = df.withColumn(
    "city",
    when(
        col("city").isin("TP.HCM", "TPHCM", "Tp.HCM", "TpHCM", "tp.hcm", "tphcm"),
        "TP.HCM"
    ).otherwise(col("city"))
)

In [16]:
df.groupBy("city").count().orderBy("count", ascending=False).show(50, truncate=False)

+-------------------------------+------+
|city                           |count |
+-------------------------------+------+
|TP.HCM                         |500742|
|CHDV 18 phòng                  |1     |
|3 tầng                         |1     |
|Góc Ôtô                        |1     |
|20x30                          |1     |
|53m2                           |1     |
|nhà mặt tiền đường số 96m chỉ 8|1     |
|lô gốc 234m2                   |1     |
|4 lầu chỉ 9                    |1     |
|Q11 .Ngang  khủng 9m x17 chỉ 23|1     |
|2 lầu  51                      |1     |
|Tân Lập 2 - 8                  |1     |
|Linh chiểu                     |1     |
|Thủ Đức                        |1     |
+-------------------------------+------+



In [17]:
df = df.filter(col("city") == "TP.HCM")

In [18]:
df.groupBy("city").count().orderBy("count", ascending=False).show(50, truncate=False)

+------+------+
|city  |count |
+------+------+
|TP.HCM|500742|
+------+------+



Chuẩn hóa Distric

In [19]:
from pyspark.sql.functions import col, when

df = df.withColumn(
    "district",
    when(col("district") == "TP. Thủ Đức", "Quận Thủ Đức (TP. Thủ Đức)")
    .when(col("district") == "Quận Thủ Đức", "Quận Thủ Đức (TP. Thủ Đức)")
    .when(col("district") == "Quận Thủ Đức (TP. Thủ Đức)", "Quận Thủ Đức (TP. Thủ Đức)")
    .otherwise(col("district"))
)

In [20]:
df.groupBy("district").count().orderBy("district").show(100, truncate=False)

+--------------------------+-----+
|district                  |count|
+--------------------------+-----+
|Huyện Bình Chánh          |6435 |
|Huyện Cần Giờ             |220  |
|Huyện Củ Chi              |3872 |
|Huyện Hóc Môn             |3720 |
|Huyện Nhà Bè              |8597 |
|Quận 1                    |26624|
|Quận 10                   |21073|
|Quận 11                   |7454 |
|Quận 12                   |16908|
|Quận 2                    |578  |
|Quận 2 (TP. Thủ Đức)      |24485|
|Quận 3                    |20436|
|Quận 4                    |11000|
|Quận 5                    |7387 |
|Quận 6                    |10360|
|Quận 7                    |41854|
|Quận 8                    |13790|
|Quận 9                    |111  |
|Quận 9 (TP. Thủ Đức)      |22068|
|Quận Ba Đình              |1    |
|Quận Bình Thạnh           |50488|
|Quận Bình Tân             |24970|
|Quận Gò Vấp               |37722|
|Quận Phú Nhuận            |26805|
|Quận Thủ Đức (TP. Thủ Đức)|18153|
|Quận Tân Bình      

In [21]:
from pyspark.sql.functions import col
df = df.filter(col("district") != "Quận Ba Đình")

In [22]:
df.groupBy("district") \
  .count() \
  .orderBy("district") \
  .show(100, truncate=False)

+--------------------------+-----+
|district                  |count|
+--------------------------+-----+
|Huyện Bình Chánh          |6435 |
|Huyện Cần Giờ             |220  |
|Huyện Củ Chi              |3872 |
|Huyện Hóc Môn             |3720 |
|Huyện Nhà Bè              |8597 |
|Quận 1                    |26624|
|Quận 10                   |21073|
|Quận 11                   |7454 |
|Quận 12                   |16908|
|Quận 2                    |578  |
|Quận 2 (TP. Thủ Đức)      |24485|
|Quận 3                    |20436|
|Quận 4                    |11000|
|Quận 5                    |7387 |
|Quận 6                    |10360|
|Quận 7                    |41854|
|Quận 8                    |13790|
|Quận 9                    |111  |
|Quận 9 (TP. Thủ Đức)      |22068|
|Quận Bình Thạnh           |50488|
|Quận Bình Tân             |24970|
|Quận Gò Vấp               |37722|
|Quận Phú Nhuận            |26805|
|Quận Thủ Đức (TP. Thủ Đức)|18153|
|Quận Tân Bình             |64233|
|Quận Tân Phú       

Chuẩn hóa Ward

In [23]:
df.groupBy("ward") \
  .count() \
  .orderBy("ward") \
  .show(100, truncate=False)

+----------------------------+-----+
|ward                        |count|
+----------------------------+-----+
|NULL                        |963  |
|Phường 1                    |12374|
|Phường 10                   |15506|
|Phường 11                   |17815|
|Phường 12                   |26198|
|Phường 13                   |20412|
|Phường 14                   |20953|
|Phường 15                   |18291|
|Phường 16                   |5722 |
|Phường 17                   |4933 |
|Phường 18                   |784  |
|Phường 19                   |978  |
|Phường 2                    |17114|
|Phường 21                   |1349 |
|Phường 22                   |3694 |
|Phường 24                   |2009 |
|Phường 25                   |6496 |
|Phường 26                   |3994 |
|Phường 27                   |421  |
|Phường 28                   |204  |
|Phường 3                    |12089|
|Phường 4                    |13256|
|Phường 5                    |16932|
|Phường 6                    |10592|
|

In [24]:
from pyspark.sql.functions import countDistinct
df.select(countDistinct("ward")).show()

+--------------------+
|count(DISTINCT ward)|
+--------------------+
|                 184|
+--------------------+



Đổi Null Ward thành Others 

In [25]:
from pyspark.sql.functions import coalesce, lit

df = df.withColumn(
    "ward",
    coalesce(col("ward"), lit("Others"))
)

Chuẩn hóa Street

Thay Null

In [26]:
from pyspark.sql.functions import coalesce, lit, col
df = df.withColumn(
    "street",
    coalesce(col("street"), lit("Others"))
)

Chuẩn hóa các tiền tố tên đường

In [27]:
from pyspark.sql.functions import regexp_replace
df = (
    df
    .withColumn(
        "street",
        regexp_replace(col("street"), r"(?i)^đ\.\s*", "Đường ")
    )
    .withColumn(
        "street",
        regexp_replace(col("street"), r"(?i)^đ\s+", "Đường ")
    )
    .withColumn(
        "street",
        regexp_replace(col("street"), r"(?i)^duong\s+", "Đường ")
    )
    .withColumn(
        "street",
        regexp_replace(col("street"), r"(?i)^đường\s+", "Đường ")
    )
)

In [28]:
from pyspark.sql.functions import regexp_replace, trim, col
df = (df
    # bỏ số nhà đầu chuỗi
    .withColumn("street",
        regexp_replace(col("street"), r"^[0-9A-Za-z./-]+\s+", "")
    )
    # bỏ mã VIC, TB, VTB...
    .withColumn("street",
        regexp_replace(col("street"), r"^(VIC\d+_\d+|VTB\d+|TB\d+)\s*", "")
    )
    # bỏ Realtor
    .withColumn("street",
        regexp_replace(col("street"), r"\[.*?\]\s*", "")
    )
    .withColumn("street", trim(col("street")))
)

In [29]:
from pyspark.sql.functions import regexp_replace

df = df.withColumn(
    "street",
    regexp_replace(col("street"), r"^(.+?)\s+\1$", r"$1")
)
df = df.withColumn(
    "street",
    regexp_replace(
        col("street"),
        r"^(Thành phố Hồ Chí Minh|TP\.?\s*HCM|TP\.?\s*Hồ Chí Minh)\s+",
        ""
    )
)

df = df.withColumn(
    "street",
    trim(regexp_replace(col("street"), r"\s+", " "))
)

In [30]:
from pyspark.sql.functions import col, regexp_replace, trim

df = (
    df

    # Chuẩn hóa tiền tố Đường 
    .withColumn(
        "street",
        regexp_replace(col("street"), r"(?i)^(đ\.?|đường|duong)\s*", "Đường ")
    )

    #Bỏ mã 
    .withColumn(
        "street",
        regexp_replace(col("street"), r"^[A-Z]{2,}\d+(_\d+)?\s*", "")
    )

    # Bỏ Realtor
    .withColumn(
        "street",
        regexp_replace(col("street"), r"\[.*?\]\s*", "")
    )

    # Bỏ số nhà đầu chuỗi
    .withColumn(
        "street",
        regexp_replace(col("street"), r"^[A-Za-z0-9/-]+\s+", "")
    )

    # Bỏ "Số xx" 
    .withColumn(
        "street",
        regexp_replace(col("street"), r"^Số\s+\S+\s+", "")
    )

    #Bỏ tên TP
    .withColumn(
        "street",
        regexp_replace(
            col("street"),
            r"^(Thành phố Hồ Chí Minh|TP\.?\s*HCM|TP\.?\s*Hồ Chí Minh)\s+",
            ""
        )
    )

    #Bỏ tên quận đứng đầu 
    .withColumn(
        "street",
        regexp_replace(
            col("street"),
            r"^(Quận\s*\d+|Gò Vấp|Thủ Đức|Bình Thạnh|Tân Bình|Phú Nhuận|Bình Tân|Quận 7|Quận 12)\s+",
            ""
        )
    )

    # Xóa tên bị lặp
    .withColumn(
        "street",
        regexp_replace(col("street"), r"^(.+?)\s+\1$", "$1")
    )

    #Xóa khoảng trắng
    .withColumn(
        "street",
        trim(regexp_replace(col("street"), r"\s+", " "))
    )
)

In [31]:
df.select("street").distinct().orderBy("street").show(200, truncate=False)

+------------------------------------+
|street                              |
+------------------------------------+
|                                    |
|#1-2 Phan Tôn                       |
|#102 An Dương Vương                 |
|#115 Trần Đình Xu                   |
|#125-#127-#129-#131-#133 Lý Tự Trọng|
|#193 Vĩnh Viễn                      |
|#42 Lê Thánh Tôn                    |
|(218/68/15) Phạm văn hai            |
|+ 65 Đề Thám                        |
|- Tp.HCM Bùi Thị Xuân               |
|- Tp.HCM Huỳnh Văn Nghệ             |
|- Tp.HCM Nguyễn Trọng Tuyển         |
|- Tp.HCM Phan Huy Ích               |
|- Tp.HCM Phạm Văn Bạch              |
|- Tp.HCM Trường Sa                  |
|09                                  |
|1005/? Nguyễn Kiệm                  |
|11 Nguyễn Văn Thủ                   |
|118/..BẠCH ĐẰNG Bạch Đằng           |
|152/2/? Nguyễn Văn Khối             |
|16                                  |
|168/? Trần Bá Giao                  |
|188 Phạm Công Trứ       

Tồn tại những value, xoá nếu tồn tại tồn tại nếu đúng 1 cái thì xóa
Và gom lại những đường có < 5 thành group by others

In [32]:
from pyspark.sql.functions import col, count

street_count = (
    df.groupBy("street")
      .agg(count("*").alias("cnt"))
)
df = (
    df.join(street_count, "street")
      .filter(col("cnt") > 1)
      .drop("cnt")
)

In [33]:
from pyspark.sql.functions import when
df = (
    df.join(street_count, "street")
      .withColumn(
          "street",
          when(col("cnt") < 5, "Others")
          .otherwise(col("street"))
      )
      .drop("cnt")
)

In [34]:
df.select("street").distinct().orderBy("street").show(500, truncate=False)

+-------------------------------+
|street                         |
+-------------------------------+
|                               |
|16                             |
|21                             |
|339                            |
|856 Tạ Quang Bửu               |
|An                             |
|Anh                            |
|Biên                           |
|Bà Cả                          |
|Bà Giang                       |
|Bà Hom                         |
|Bà Huyện Thanh Quan            |
|Bà Hạt                         |
|Bà Khoan                       |
|Bà Ký                          |
|Bà Lài                         |
|Bà Lê Chân                     |
|Bà Thiên                       |
|Bà Triệu                       |
|Bà Trưng                       |
|Bà Xán                         |
|Bà Điểm                        |
|Bà Điểm 10                     |
|Bà Điểm 11                     |
|Bà Điểm 4                      |
|Bà Điểm 5                      |
|Bà Điểm 6    

Chuẩn hóa address

In [35]:
df.count()

500235

Chuẩn Hóa usable_area( diện tích sử dụng)

Xóa M2

In [36]:
from pyspark.sql.functions import col, regexp_replace, trim

df = (
    df
    .withColumn(
        "usable_area",
        trim(
            regexp_replace(col("usable_area"), r"(?i)\s*m2\s*$", "")
        )
    )
    .withColumn(
        "usable_area",
        regexp_replace(col("usable_area"), ",", ".").cast("double")
    )
)

Lọc từ xuất hiện ()

In [37]:
from pyspark.sql.functions import when, col
df = df.withColumn(
    "usable_area",
    when(col("usable_area").contains("Báo vi phạm"), None)
    .otherwise(col("usable_area"))
)

In [38]:
df.count()

500235

Xử lý land_area ( diện tích đất )

Giữ cột gốc và lấy diện tích

In [39]:
from pyspark.sql.functions import col, regexp_extract, regexp_replace
df = df.withColumnRenamed("land_area", "land_area_raw")
df = df.withColumn(
    "land_area",
    regexp_replace(
        regexp_extract(col("land_area_raw"), r"([\d.,]+)\s*m", 1),
        ",",
        "."
    ).cast("double")
)

In [40]:
df.count()

500235

Tách chiều dài và chiều rộng 

In [41]:
from pyspark.sql.functions import col, regexp_extract, regexp_replace
df = df.withColumn(
    "width",
    regexp_replace(
        regexp_extract(col("land_area_raw"), r"\(([\d.,]+)\s*x", 1),
        ",",
        "."
    ).cast("double")
)

In [42]:
from pyspark.sql.functions import col, regexp_extract, regexp_replace

df = df.withColumn(
    "length",
    regexp_replace(
        regexp_extract(col("land_area_raw"), r"x\s*([\d.,]+)", 1),
        ",",
        "."
    ).cast("double")
)

Xử lý usable_area và land_area

In [43]:
from pyspark.sql.functions import col, coalesce

# Xóa những dòng mà cả usable_area và land_area đều NULL
df = df.filter(
    ~(col("usable_area").isNull() & col("land_area").isNull())
)

In [44]:
df.count()

499478

Đã xử lý được vấn đề NULL nhưng chưa xử lý những vấn đề trường hợp tồn tại giá trị cột này nhưng cột kia không có giá trị.

Xử lý URL Tách lấy category

Tách nhà ở ra thành các Sub

In [45]:
from pyspark.sql.functions import when, col

df = (
    df.withColumn(
        "subcategory",
        when(
            col("url").contains("/mua-nha-mat-tien-pho"),
            "Mặt tiền phố"
        )
        .when(
            col("url").contains("/mua-nha-biet-thu-lien-ke"),
            "Biệt thự - Liền kề"
        )
        .when(
            col("url").contains("/mua-duong-noi-bo"),
            "Đường nội bộ"
        )
        .when(
            col("url").contains("/mua-nha-hem-ngo"),
            "Hẻm - Ngõ"
        )
        .otherwise(None)
    )
)

In [46]:
from pyspark.sql.functions import when, col

df = (
    df.withColumn(
        "category",
        when(col("subcategory").isNotNull(), "Nhà ở")
        .otherwise(None)
    )
)

Tách căn hộ ra thành các Sub

In [47]:
from pyspark.sql.functions import when, col
df = (
    df
    .withColumn(
        "subcategory",
        when(
            col("subcategory").isNotNull(),
            col("subcategory")
        )
        .when(
            col("url").contains("/mua-can-ho-chung-cu"),
            "Chung cư"
        )
        .when(
            col("url").contains("/mua-can-ho-tap-the-cu-xa"),
            "Tập thể - Cư xá"
        )
        .when(
            col("url").contains("/mua-can-ho-penthouse"),
            "Penthouse"
        )
        .when(
            col("url").contains("/mua-can-ho-dich-vu"),
            "Dịch vụ"
        )
        .when(
            col("url").contains("/mua-can-ho-officetel"),
            "Officetel"
        )
        .otherwise(col("subcategory"))
    )
)

In [48]:
from pyspark.sql.functions import when, col
df = (
    df
    .withColumn(
        "category",
        when(
            col("category").isNotNull(),
            col("category")
        )
        .when(
            col("subcategory").isin(
                "Chung cư",
                "Tập thể - Cư xá",
                "Penthouse",
                "Dịch vụ",
                "Officetel"
            ),
            "Căn hộ"
        )
        .otherwise(col("category"))
    )
)

Tách đất ra thành các Sub

In [49]:
from pyspark.sql.functions import when, col

df = (
    df
    .withColumn(
        "subcategory",
        when(col("subcategory").isNotNull(), col("subcategory"))
        .when(col("url").contains("/mua-dat-tho-cu"), "Đất thổ cư")
        .when(col("url").contains("/mua-dat-nen-du-an"), "Đất nền dự án")
        .when(col("url").contains("/mua-dat-nong-nghiep"), "Đất nông nghiệp")
        .when(col("url").contains("/mua-dat-kho-xuong"), "Đất kho xưởng")
        .otherwise(col("subcategory"))
    )
)

In [50]:
from pyspark.sql.functions import when, col
df = (
    df
    .withColumn(
        "category",
        when(col("category").isNotNull(), col("category"))
        .when(
            col("subcategory").isin(
                "Đất thổ cư",
                "Đất nền dự án",
                "Đất nông nghiệp",
                "Đất kho xưởng"
            ),
            "Đất"
        )
        .otherwise(col("category"))
    )
)

Táăt mặt bằng ra thành các Sub

In [51]:
from pyspark.sql.functions import when, col
df = (
    df
    .withColumn(
        "subcategory",
        when(col("subcategory").isNotNull(), col("subcategory"))

        .when(
            col("url").contains("/mua-mat-bang-cua-hang-shop-quan-an-nha-hang"),
            "Quán ăn - Nhà hàng"
        )
        .when(
            col("url").contains("/mua-mat-bang-cua-hang-shop-cafe-do-uong"),
            "Cafe - Đồ uống"
        )
        .when(
            col("url").contains("/mua-mat-bang-cua-hang-shop-thoi-trang-my-pham-thuoc"),
            "Thời trang - Mỹ phẩm - Nhà thuốc"
        )
        .when(
            col("url").contains("/mua-mat-bang-cua-hang-shop-spa-tiem-toc-nail"),
            "Spa - Tiệm tóc - Nail"
        )
        .when(
            col("url").contains("/mua-cua-hang-shop-shophouse"),
            "Shophouse"
        )
        .when(
            col("url").contains("/mua-mat-bang-cua-hang-shop-nhieu-muc-dich"),
            "Nhiều mục đích"
        )

        .otherwise(col("subcategory"))
    )
)

In [52]:
from pyspark.sql.functions import when, col

df = (
    df
    .withColumn(
        "category",
        when(col("category").isNotNull(), col("category"))
        .when(
            col("subcategory").isin(
                "Quán ăn - Nhà hàng",
                "Cafe - Đồ uống",
                "Thời trang - Mỹ phẩm - Nhà thuốc",
                "Spa - Tiệm tóc - Nail",
                "Shophouse",
                "Nhiều mục đích"
            ),
            "Mặt bằng"
        )
        .otherwise(col("category"))
    )
)

In [53]:
from pyspark.sql.functions import count, sum, col

df.groupBy("category").agg(
    count("*").alias("total"),
    sum(col("usable_area").isNull().cast("int")).alias("usable_null"),
    sum(col("land_area").isNull().cast("int")).alias("land_null")
).filter(col("category").isNotNull()).show(truncate=False)

+--------+------+-----------+---------+
|category|total |usable_null|land_null|
+--------+------+-----------+---------+
|Nhà ở   |378854|211951     |23       |
|Mặt bằng|1173  |0          |1170     |
|Đất     |30477 |12375      |0        |
|Căn hộ  |88973 |6          |88786    |
+--------+------+-----------+---------+



1. Căn hộ
Giữ nguyên land_area = NULL.
Không fill.
2. Đất
Giữ nguyên usable_area = NULL.
Không fill.
3. Mặt bằng
Giữ nguyên land_area = NULL.
Không fill.
4. Nhà ở
377k bản ghi.
Chỉ 23 bản ghi thiếu land_area.
Nhưng 211k bản ghi thiếu usable_area.
=> Giữ nguyên hai cột. Thêm một cột mới: arena

-Nhà có usable_area → dùng usable_area.

-Nhà không có usable_area → dùng land_area.

-Căn hộ → tự động dùng usable_area.

-Đất → tự động dùng land_area.

-Mặt bằng → tự động dùng usable_area.

Xóa Null

In [54]:
from pyspark.sql.functions import col, when
df = df.withColumn(
    "area",
    when(
        col("category") == "Nhà ở",
        when(col("usable_area").isNotNull(), col("usable_area"))
        .otherwise(col("land_area"))
    )
    .when(
        col("category") == "Căn hộ",
        col("usable_area")
    )
    .when(
        col("category") == "Đất",
        col("land_area")
    )
    .when(
        col("category") == "Mặt bằng",
        col("usable_area")
    )
    .otherwise(None)
)

In [55]:
df.count()


499478

Xử lý null bedroom và bathroom Đất (0,0)

Chuyển bedrooms bathrooms thành int

In [56]:
from pyspark.sql.functions import col

df = (
    df
    .withColumn("bedrooms", col("bedrooms").cast("int"))
    .withColumn("bathrooms", col("bathrooms").cast("int"))
)

Đất không có phòng ngủ, không có toilet.
=> Gán:
bedroom = 0
bathroom = 0

In [57]:
from pyspark.sql.functions import when, col

df = (
    df
    .withColumn(
        "bedrooms",
        when(
            (col("category") == "Đất") & col("bedrooms").isNull(),
            0
        ).otherwise(col("bedrooms"))
    )
    .withColumn(
        "bathrooms",
        when(
            (col("category") == "Đất") & col("bathrooms").isNull(),
            0
        ).otherwise(col("bathrooms"))
    )
)

Mặt bằng kinh doanh (shop, cửa hàng, quán ăn, shophouse...) không được mô tả bằng số phòng ngủ.
bedroom = 0 => batroom = 0

In [58]:
from pyspark.sql.functions import col, coalesce
df = df.withColumn(
    "area",
    coalesce(col("usable_area"), col("land_area"))
)

In [59]:
df.filter(col("category") == "Mặt bằng").groupBy("bathrooms").count().show()

+---------+-----+
|bathrooms|count|
+---------+-----+
|        2|    1|
|     NULL| 1172|
+---------+-----+



Điều này cho thấy gần như toàn bộ tin đăng không khai báo số phòng ngủ và phòng tắm. Với loại bất động sản là Mặt bằng, đây là điều hợp lý vì các thông tin này thường không phải đặc trưng chính.
bedroom: gán 0 cho tất cả giá trị NULL.
bathroom: chỉ gán 0 cho các giá trị NULL, giữ nguyên bản ghi có bathroom = 2.

In [60]:
from pyspark.sql.functions import when, col
df = (
    df
    .withColumn(
        "bedrooms",
        when(
            (col("category") == "Mặt bằng") & col("bedrooms").isNull(),
            0
        ).otherwise(col("bedrooms"))
    )
    .withColumn(
        "bathrooms",
        when(
            (col("category") == "Mặt bằng") & col("bathrooms").isNull(),
            0
        ).otherwise(col("bathrooms"))
    )
)

Căn hộ

In [61]:
from pyspark.sql.functions import count, sum, col

df.filter(col("category") == "Căn hộ").agg(
    count("*").alias("total"),
    sum(col("bedrooms").isNull().cast("int")).alias("bedroom_null"),
    sum(col("bathrooms").isNull().cast("int")).alias("bathroom_null")
).show()

+-----+------------+-------------+
|total|bedroom_null|bathroom_null|
+-----+------------+-------------+
|88973|         384|          614|
+-----+------------+-------------+



In [62]:
df.filter(
    (col("category") == "Căn hộ") &
    col("bedrooms").isNotNull()
).groupBy("bedrooms").count().orderBy("bedrooms").show()

+--------+-----+
|bedrooms|count|
+--------+-----+
|       1| 8103|
|       2|60819|
|       3|17500|
|       4| 1106|
|       5|  190|
|       6|  104|
|       7|   23|
|       8|   52|
|       9|   33|
|      10|   76|
|      11|   40|
|      12|   40|
|      13|   30|
|      14|   22|
|      15|   45|
|      16|   16|
|      17|   14|
|      18|   31|
|      19|   11|
|      20|   40|
+--------+-----+
only showing top 20 rows



In [63]:
df.filter(
    (col("category") == "Căn hộ") &
    col("bathrooms").isNotNull()
).groupBy("bathrooms").count().orderBy("bathrooms").show()

+---------+-----+
|bathrooms|count|
+---------+-----+
|        1|15189|
|        2|68731|
|        3| 2838|
|        4|  617|
|        5|  146|
|        6|   70|
|        7|   32|
|        8|   35|
|        9|   31|
|       10|   77|
|       11|   44|
|       12|   38|
|       13|   34|
|       14|   24|
|       15|   38|
|       16|   27|
|       17|   17|
|       18|   30|
|       19|   11|
|       20|   44|
+---------+-----+
only showing top 20 rows




Mode bedrooms = 2 (60.819 bản ghi)
Mode bathrooms = 2 (68.731 bản ghi)
Tỷ lệ thiếu:
bedrooms: 384 / 88.973 = 0,43%
bathrooms: 614 / 88.973 = 0,69%

=>  mode = 2


In [64]:
from pyspark.sql.functions import when, col

df = (
    df
    .withColumn(
        "bedrooms",
        when(
            (col("category") == "Căn hộ") & col("bedrooms").isNull(),
            2
        ).otherwise(col("bedrooms"))
    )
    .withColumn(
        "bathrooms",
        when(
            (col("category") == "Căn hộ") & col("bathrooms").isNull(),
            2
        ).otherwise(col("bathrooms"))
    )
)

Nhà ở

In [65]:
from pyspark.sql.functions import count, sum, col

df.filter(col("category") == "Nhà ở").agg(
    count("*").alias("total"),
    sum(col("bedrooms").isNull().cast("int")).alias("bedrooms_null"),
    sum(col("bathrooms").isNull().cast("int")).alias("bathrooms_null")
).show()

+------+-------------+--------------+
| total|bedrooms_null|bathrooms_null|
+------+-------------+--------------+
|378854|        11182|         11220|
+------+-------------+--------------+



In [66]:
df.filter(
    (col("category") == "Nhà ở") &
    col("bedrooms").isNotNull()
).groupBy("bedrooms").count().orderBy("bedrooms").show()

+--------+-----+
|bedrooms|count|
+--------+-----+
|       1|20052|
|       2|75736|
|       3|84754|
|       4|84297|
|       5|36491|
|       6|23692|
|       7| 9266|
|       8| 7939|
|       9| 2864|
|      10| 5871|
|      11| 1069|
|      12| 2171|
|      13|  555|
|      14| 1174|
|      15| 1345|
|      16|  881|
|      17|  349|
|      18|  781|
|      19|  257|
|      20| 1282|
+--------+-----+
only showing top 20 rows



In [67]:
df.filter(
    (col("category") == "Nhà ở") &
    col("bathrooms").isNotNull()
).groupBy("bathrooms").count().orderBy("bathrooms").show()

+---------+-----+
|bathrooms|count|
+---------+-----+
|        1|27398|
|        2|92016|
|        3|80290|
|        4|63058|
|        5|43091|
|        6|20852|
|        7| 7675|
|        8| 6572|
|        9| 3916|
|       10| 4950|
|       11| 1867|
|       12| 1998|
|       13|  827|
|       14| 1058|
|       15| 1176|
|       16| 1041|
|       17|  532|
|       18|  663|
|       19|  391|
|       20| 1126|
+---------+-----+
only showing top 20 rows



KHÔNG DÙNG MODE ĐƯỢC 
bedrooms mode = 4 (84.297), nhưng 3 (84.754) gần như ngang nhau, 2 (75.736) cũng rất nhiều.
bathrooms mode = 2 (92.016), nhưng 3 (80.290) và 4 (63.058) cũng rất phổ biến.

Điền tất cả NULL = 4 hoặc 2 sẽ làm mất đặc trưng của dữ liệu.
Sau khi xử lý theo từng loại bất động sản, số giá trị thiếu của bedrooms và bathrooms còn khoảng 3% tổng dữ liệu. Do không thể suy luận chính xác số phòng từ các thuộc tính còn lại và việc thay thế bằng mode/median có thể làm sai lệch phân bố dữ liệu.
KNN thì tốn tài nguyên để chạy

In [68]:
df.count()

499478

Xử lý vấn đề outlier của Price 

Nếu dùng IQR Bình thường thì xảy ra vấn đề => các khu giá cao quận 1 hoặc kho xưởng sẽ bị loại bỏ 

=> Group by theo quận + cate để loại Outlier. Tuy nhiên sẽ xảy ra vấn đề nếu area cao thì dể đánh outlier => Tạo cột 
price_per_m2 = price / area

In [69]:
from pyspark.sql import functions as F
from pyspark.sql.functions import broadcast

# Chỉ giữ dữ liệu hợp lệ để tính price_per_m2
df = df.filter(
    (F.col("price").isNotNull()) &
    (F.col("area").isNotNull()) &
    (F.col("price") > 0) &
    (F.col("area") > 0)
)

# Tính giá / m2
df = df.withColumn(
    "price_per_m2",
    F.col("price") / F.col("area")
)

# Tính Q1, Q3 theo Category + District
iqr_df = (
    df.groupBy("category", "district")
      .agg(
          F.expr(
              "percentile_approx(price_per_m2, array(0.25, 0.75), 10000)"
          ).alias("quartiles")
      )
      .select(
          "category",
          "district",
          F.col("quartiles")[0].alias("Q1"),
          F.col("quartiles")[1].alias("Q3")
      )
      .withColumn("IQR", F.col("Q3") - F.col("Q1"))
      .withColumn("Lower", F.col("Q1") - 1.5 * F.col("IQR"))
      .withColumn("Upper", F.col("Q3") + 1.5 * F.col("IQR"))
)

# Join lại
df = df.join(
    broadcast(iqr_df),
    ["category", "district"],
    "left"
)

# Loại bỏ outlier
df = df.filter(
    (F.col("price_per_m2") >= F.col("Lower")) &
    (F.col("price_per_m2") <= F.col("Upper"))
)

# Xóa cột tạm
df = df.drop(
    "Q1",
    "Q3",
    "IQR",
    "Lower",
    "Upper"
)

In [70]:
df.count()

478070

Đối với dữ liệu bất động sản dân dụng (nhà ở, căn hộ, biệt thự, nhà phố) và một phần bất động sản thương mại, số lượng phòng ngủ hoặc phòng vệ sinh lớn hơn 40 rất hiếm gặp và chủ yếu xuất hiện ở các khách sạn, tòa nhà hoặc do lỗi nhập liệu. Để giảm ảnh hưởng của các giá trị bất thường đến quá trình phân tích, các bản ghi có bedrooms > 40 hoặc bathrooms > 40 được xem là outlier và loại bỏ. 

Vì sao là 40 Các tòa nhà văn phòng cao tầng tại TP.HCM thường có khoảng 15–25 tầng. Giả sử trung bình mỗi tầng có 2 phòng ngủ/phòng chức năng và 2 phòng vệ sinh thì số lượng phòng ngủ hoặc phòng tắm hiếm khi vượt quá 40.


In [71]:
from pyspark.sql.functions import col

df = df.filter(
    (
        col("bedrooms").isNull() |
        (col("bedrooms") <= 40)
    ) &
    (
        col("bathrooms").isNull() |
        (col("bathrooms") <= 40)
    )
)

In [72]:
from pyspark.sql.functions import col

df = df.filter(
    (
        col("usable_area").isNull() |
        (col("usable_area") > 7)
    ) &
    (
        col("land_area").isNull() |
        (col("land_area") > 7)
    )
)

Business Rule nhằm loại bỏ các bản ghi có giá bán dưới 100 triệu đồng. Ngưỡng này được lựa chọn dựa trên đặc điểm thị trường bất động sản TP.HCM, vì các giao dịch mua bán dưới mức giá này hầu như không tồn tại và thường là dữ liệu nhập sai, giá tượng trưng hoặc tin đăng không hợp lệ.

In [79]:
df = df.filter(
    (F.col("price") >= 100000000) &
    (F.col("price").isNotNull())
)

In [80]:
df.count()

473994

Tách ngày

In [73]:
from pyspark.sql.functions import to_date, col

df = df.withColumn(
    "posted_date",
    to_date(col("posted_date"), "dd/MM/yyyy")
)

In [74]:
df.select("posted_date").show(5, False)

+-----------+
|posted_date|
+-----------+
|2022-03-22 |
|2022-03-22 |
|2022-03-22 |
|2022-03-21 |
|2022-03-21 |
+-----------+
only showing top 5 rows



In [ ]:
df = df.select(
    "property_id",
    "title",
    "url",
    "category",
    "subcategory",
    "city",
    "district",
    "ward",
    "street",
    "address",
    "price",
    "usable_area",
    "land_area",
    "width",
    "length",
    "area",
    "price_per_m2",
    "bedrooms",
    "bathrooms",
    "legal_status",
    "posted_date",
    "land_area_raw"
)

In [76]:
from pyspark.sql import functions as F

print(f"Total records: {df.count()}")

print("\nSchema:")
df.printSchema()    


Total records: 476160

Schema:
root
 |-- property_id: string (nullable = true)
 |-- title: string (nullable = true)
 |-- url: string (nullable = true)
 |-- category: string (nullable = true)
 |-- subcategory: string (nullable = true)
 |-- city: string (nullable = true)
 |-- district: string (nullable = true)
 |-- ward: string (nullable = false)
 |-- street: string (nullable = false)
 |-- address: string (nullable = true)
 |-- price: long (nullable = true)
 |-- usable_area: double (nullable = true)
 |-- land_area: double (nullable = true)
 |-- width: double (nullable = true)
 |-- length: double (nullable = true)
 |-- area: double (nullable = true)
 |-- price_per_m2: double (nullable = true)
 |-- bedrooms: integer (nullable = true)
 |-- bathrooms: integer (nullable = true)
 |-- legal_status: string (nullable = true)
 |-- posted_date: date (nullable = true)
 |-- land_area_raw: string (nullable = true)



Test Parquet D

Ghi vào Minio Silver

In [81]:
df.write.mode("overwrite").parquet(
    "s3a://housing-data/silver/mogi_clean.parquet"
)

# Không chạy Cell dưới, Cell dưới mục đích test data mỗi khi clean data

In [ ]:
#Không chạy cell này
import pandas as pd

pdf = df.toPandas()

pdf.to_csv(
    "mogi_clean.csv",
    index=False,
    encoding="utf-8-sig"
)

print(" mogi_clean.csv")
print(f"Số dòng: {len(pdf):,}")
print(f"Số cột : {len(pdf.columns)}")

Đã xuất thành công: mogi_clean.csv
Số dòng: 473,994
Số cột : 22
